In [2]:
import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc

In [3]:
adata = ad.read_h5ad("e4ddac12-f48f-4455-8e8d-c2a48a683437.h5ad")
adata

AnnData object with n_obs × n_vars = 129495 × 29322
    obs: 'Class', 'CrossArea_subclass', 'CrossArea_cluster', 'WithinArea_subclass', 'WithinArea_cluster', 'Source', 'Layer', 'Location', 'Region', 'Subregion', 'nCount_RNA', 'nFeature_RNA', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'sex_ontology_term_id', 'suspension_type', 'tissue_ontology_term_id', 'is_primary_data', 'donor_id', 'tissue_type', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid'
    var: 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type'
    uns: 'batch_condition', 'citation', 'default_embedding', 'is_primary_data', 'organism', 'organism_ontology_term_id', 'schema_reference', 'schema_version', 'title'
    obsm: 'X_UMAP'

In [4]:
adata.X.data

array([5.212478 , 5.212478 , 5.212478 , ..., 4.22125  , 5.3100276,
       5.3100276], dtype=float32)

In [5]:
adata.raw.X.data

array([1., 1., 1., ..., 1., 3., 3.], dtype=float32)

## figure out what is adata.X.data

In [5]:
# identify expressed genes in .raw

# Mean expression per gene
mean_expr = np.array(adata.X.mean(axis=0)).ravel()

gene_expr = pd.DataFrame({
    "gene": adata.var["feature_name"],
    "mean_expression": mean_expr
})

# Sort by expression
gene_expr_sorted = gene_expr.sort_values(
    "mean_expression", ascending=False
)

# Top 10 most expressed genes
gene_expr_sorted.head(10)

,gene,mean_expression
ENSG00000251562,MALAT1,10.660508
ENSG00000178568,ERBB4,8.681052
ENSG00000174469,CNTNAP2,8.620510
ENSG00000185736,ADARB2,8.525735
ENSG00000021645,NRXN3,8.337770
ENSG00000179915,NRXN1,8.198131
ENSG00000102466,FGF14,8.125409
ENSG00000189337,KAZN,8.107231
ENSG00000185737,NRG3,8.051261
ENSG00000175161,CADM2,7.888232


## clip at 1

In [5]:
adata.X.data = np.minimum(adata.X.data, 1)
adata.raw._X.data = np.minimum(adata.raw._X.data, 1)

In [6]:
new_adata = ad.AnnData(
    X=adata.raw._X.copy(),     # use the modified raw
    obs=adata.obs.copy(),      # keep cell metadata
    var=adata.var.copy()       # keep gene metadata
)

In [7]:
adata.write_h5ad("e4ddac12-f48f-4455-8e8d-c2a48a683437-allones.h5ad")

## clip at 2

In [10]:
adata.X.data = np.minimum(adata.X.data, 2)
adata.raw._X.data = np.minimum(adata.raw._X.data, 2)

In [11]:
new_adata = ad.AnnData(
    X=adata.raw._X.copy(),     # use the modified raw
    obs=adata.obs.copy(),      # keep cell metadata
    var=adata.var.copy()       # keep gene metadata
)

In [12]:
new_adata.write_h5ad("e4ddac12-f48f-4455-8e8d-c2a48a683437-cut2.h5ad")

## log1p

In [21]:
adata.X.data = np.log1p(adata.X.data)
adata.raw._X.data = np.log1p(adata.raw._X.data)

In [22]:
adata.raw._X.data

array([0.6931472, 0.6931472, 0.6931472, ..., 0.6931472, 1.3862944,
       1.3862944], dtype=float32)

In [6]:
new_adata = ad.AnnData(
    X=adata.raw._X.copy(),     # use the modified raw
    obs=adata.obs.copy(),      # keep cell metadata
    var=adata.var.copy()       # keep gene metadata
)

In [7]:
new_adata.write_h5ad("e4ddac12-f48f-4455-8e8d-c2a48a683437-log1p.h5ad")

## 50% variation 

In [5]:
noise_factors = np.random.choice([0.5, 1.5], size=adata.X.data.shape)

In [6]:
adata.X.data = adata.X.data * noise_factors

In [7]:
adata.raw._X.data = adata.raw._X.data * noise_factors

In [8]:
new_adata = ad.AnnData(
    X=adata.raw._X.copy(),     # use the modified raw
    obs=adata.obs.copy(),      # keep cell metadata
    var=adata.var.copy()       # keep gene metadata
)

In [9]:
new_adata.write_h5ad("e4ddac12-f48f-4455-8e8d-c2a48a683437-50percentNoise.h5ad")

## 200 (1% all genes) genes mask out

In [5]:
n_genes_to_zero = 200
genes_idx = np.random.choice(adata.X.shape[1], size=n_genes_to_zero, replace=False)
genes_idx

array([ 2353, 16265,  8667, 12690,  8425, 21234, 15512,  6043,  8598,
       15863,  5798, 28326,  3695, 27299,  3431, 23005, 28455,  2790,
       28097,   913,   995, 21761, 17198,  8581, 27990, 20561, 14293,
       14895,  8399,  9194,  2765, 28523, 21895, 20094,  8986, 15528,
        9046, 25163, 14251, 18758, 21282, 21927, 12899,  2531, 14639,
       15584,  1554, 10491,  6273, 21320,  7798,  1135,   604,  2504,
       27833, 23921, 14888, 21954, 26140,  4641, 16958,  6502, 14797,
       16555,  6437,  3078,  9652, 11623,  1362, 12104, 26014, 19411,
        5780, 19569,  7921, 20676, 17862, 21941, 27380, 18257, 21952,
       14396, 15074, 24040, 10717, 23504, 21982,  8009,    77, 16475,
       12090, 10449, 18627,  3489, 24335, 27309, 10142, 27978,  4993,
       11225, 12925, 10933, 25608, 12854, 14182, 21886,  1964,  9887,
       20024,  8072, 15306, 25803,  6281, 10047, 17125, 16762,  4658,
        4922, 24931, 16253, 18366,  9589, 26191,  1285, 20415, 19796,
        6783, 10807,

In [6]:
adata.X[:, genes_idx] = 0 
adata.raw._X[:, genes_idx] = 0

/home/ec2-user/.local/lib/python3.9/site-packages/scipy/sparse/_index.py:151: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil_matrix is more efficient.
  self._set_arrayXarray(i, j, x)


In [7]:
new_adata = ad.AnnData(
    X=adata.raw._X.copy(),     # use the modified raw
    obs=adata.obs.copy(),      # keep cell metadata
    var=adata.var.copy()       # keep gene metadata
)

In [8]:
new_adata.write_h5ad("e4ddac12-f48f-4455-8e8d-c2a48a683437-200geneMask.h5ad")

## 1% (high and moderately) expressed genes mask out

In [5]:
# identify expressed genes in .raw

# Mean expression per gene
mean_expr = np.array(adata.raw.X.mean(axis=0)).ravel()

gene_expr = pd.DataFrame({
    "gene": adata.var["feature_name"],
    "mean_expression": mean_expr
})

# Sort by expression
gene_expr_sorted = gene_expr.sort_values(
    "mean_expression", ascending=False
)

# Top 10 most expressed genes
gene_expr_sorted.head(10)

,gene,mean_expression
ENSG00000251562,MALAT1,655.041016
ENSG00000178568,ERBB4,102.441399
ENSG00000174469,CNTNAP2,101.208511
ENSG00000185736,ADARB2,88.859177
ENSG00000021645,NRXN3,74.327271
ENSG00000183117,CSMD1,66.087799
ENSG00000078328,RBFOX1,61.687180
ENSG00000185008,ROBO2,61.423500
ENSG00000102466,FGF14,60.291084
ENSG00000179915,NRXN1,59.872379


In [6]:
n_genes_expressed = (gene_expr_sorted["mean_expression"] > 1).sum()
n_genes_expressed, gene_expr_sorted[gene_expr_sorted["mean_expression"] > 1]

(np.int64(2899),
                     gene  mean_expression
 ENSG00000251562   MALAT1       655.041016
 ENSG00000178568    ERBB4       102.441399
 ENSG00000174469  CNTNAP2       101.208511
 ENSG00000185736   ADARB2        88.859177
 ENSG00000021645    NRXN3        74.327271
 ...                  ...              ...
 ENSG00000113595   TRIM23         1.002501
 ENSG00000154447   SH3RF1         1.000657
 ENSG00000168734     PKIG         1.000610
 ENSG00000177453    NIM1K         1.000598
 ENSG00000101972    STAG2         1.000529
 
 [2899 rows x 2 columns])

In [8]:
# randomly select 0.5% from moderately to high experssed genes
selected_genes = gene_expr_sorted[
    gene_expr_sorted["mean_expression"] > 1
].sample(n= int(n_genes_expressed * 0.01), random_state=42)
selected_genes

,gene,mean_expression
ENSG00000206579,XKR4,10.706738
ENSG00000165124,SVEP1,1.761285
ENSG00000163625,WDFY3,4.630690
ENSG00000180776,ZDHHC20,1.627769
ENSG00000085382,HACE1,1.255332
ENSG00000070214,SLC44A1,1.878551
ENSG00000149269,PAK1,2.316570
ENSG00000131143,COX4I1,1.139100
ENSG00000145242,EPHA5,4.128540
ENSG00000151718,WWC2,1.417390


In [9]:
genes_to_mask = selected_genes["gene"].index

# Indices of genes in adata
gene_idx = adata.var_names.get_indexer(genes_to_mask)

# Safety check (remove genes not found)
gene_idx = gene_idx[gene_idx != -1]

gene_idx, len(gene_idx)

(array([28236, 25352, 28051, 28502,  8908, 23847, 17593,  4402,  6457,
        28205, 28718,  7776,   945,  6113, 10824, 18325, 27015,  1560,
        19761, 27325, 14890, 12617, 28265,  3454, 29028,  1022,  6580,
         1976]),
 28)

In [10]:
adata.raw._X[:, gene_idx] = 0
adata.X[:, gene_idx] = 0

/home/ec2-user/.local/lib/python3.9/site-packages/scipy/sparse/_index.py:151: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil_matrix is more efficient.
  self._set_arrayXarray(i, j, x)


In [11]:
new_adata = ad.AnnData(
    X=adata.raw._X.copy(),     # use the modified raw
    obs=adata.obs.copy(),      # keep cell metadata
    var=adata.var.copy()       # keep gene metadata
)

In [12]:
np.all(new_adata.X[:, 27325].toarray() == 0), new_adata.X[:, 27325].mean(), new_adata.raw

(np.True_, np.float32(0.0), None)

In [13]:
idx = new_adata.var.index[adata.var["feature_name"] == "XKR4"]
idx, adata[:, idx].X.mean()

(Index(['ENSG00000206579'], dtype='object'), np.float32(0.0))

In [13]:
new_adata.write_h5ad("e4ddac12-f48f-4455-8e8d-c2a48a683437-28expressedGeneMask.h5ad")

## use X , assume that's logNorm

In [5]:
adata.raw = None

In [6]:
adata.write_h5ad("e4ddac12-f48f-4455-8e8d-c2a48a683437-X.h5ad")

## logNorm using scanpy functions

In [6]:
new_adata = ad.AnnData(
    X=adata.raw._X.copy(),     # use the modified raw
    obs=adata.obs.copy(),      # keep cell metadata
    var=adata.var.copy()       # keep gene metadata
)

In [10]:
new_adata.raw

In [13]:
# Normalizing to median total counts
sc.pp.normalize_total(new_adata)
# Logarithmize the data
sc.pp.log1p(new_adata)

In [14]:
new_adata.raw

In [23]:
new_adata.X.data

array([1.2188623, 1.2188623, 1.2188623, ..., 0.6291861, 1.2887502,
       1.2887502], dtype=float32)

In [24]:
new_adata.write_h5ad("e4ddac12-f48f-4455-8e8d-c2a48a683437-logNorm.h5ad")

## logNorm target to 1M

In [6]:
new_adata = ad.AnnData(
    X=adata.raw._X.copy(),     # use the modified raw
    obs=adata.obs.copy(),      # keep cell metadata
    var=adata.var.copy()       # keep gene metadata
)
import scanpy as sc
# Normalizing to median total counts
sc.pp.normalize_total(new_adata, target_sum = 1_000_000)
# Logarithmize the data
sc.pp.log1p(new_adata)

In [8]:
new_adata.X.data

array([5.2133865, 5.2133865, 5.2133865, ..., 4.221912 , 5.3106956,
       5.3106956], dtype=float32)

In [7]:
new_adata.write_h5ad("e4ddac12-f48f-4455-8e8d-c2a48a683437-logNorm1M.h5ad")